# Phase 1: Externalized Spatial Memory & Stigmergic Cooperation (Demo)
**Project:** Externalized Memory and Institutional Rule Systems in Multi-Agent Societies  
**Authors:** Candan Tuna Mesut & Kayra Sevim  
**Supervisor:** Dr. Zeki Doruk Erden  

---

## 📌 Purpose of this Notebook
This notebook serves as the **official demonstration and entry point for Phase 1 (External Memory)**.
It isolates the final, fully-verified multi-agent environment from earlier iterations and outdated reports, allowing anyone (or automated assistants like Claude) to easily inspect and reproduce the empirical results:
- **Autonomous Stigmergic Writing ()**
- **Gradual & Replenishable Resource Pools**
- **Per-Timestep Living Cost**
- **Empirical Validation: Local Stigmergy vs. Full-Map Baseline**

In [ ]:
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
from env import MARLSurvivalEnv, ACTION_NAMES, WRITE_TO_MAP
from model import ActorCritic
from config import CONFIG

print("✓ All core modules imported successfully.")
print(f"✓ Device detected: {'mps' if torch.backends.mps.is_available() else ('cuda' if torch.cuda.is_available() else 'cpu')}")

## 1. Environment Architecture & Mechanics Verification
Let's instantiate the final environment in **Local Stigmergy Mode**.
- Grid size:  	imes 20$
- Action space: 6 actions (, , , , , )
- Observation space: 30 continuous dimensions ( 	imes 3$ local agents,  	imes 3$ local food levels,  	imes 3$ local pheromone intensity)

In [ ]:
cfg = CONFIG.copy()
cfg["VISIBILITY_MODE"] = "local"
env = MARLSurvivalEnv(config=cfg, max_steps=100)
obs, info = env.reset(seed=42)

print(f"Agents: {env.agents}")
print(f"Action space per agent: {env.action_space(env.agents[0])}")
print(f"Observation shape per agent: {obs[env.agents[0]].shape}")
print(f"Initial resource pools: {len(env.rewards_data)} pools of capacity {env.pool_capacity} units")

## 2. Load Trained Policy & Run an Evaluation Rollout
We load the trained policy checkpoint () trained for 150 episodes via Multi-Agent PPO.

In [ ]:
model_path = "marl_survival_model_local.pt"
model = ActorCritic(obs_dim=30, action_dim=6, hidden_dim=128)

if os.path.exists(model_path):
    model.load_state_dict(torch.load(model_path, map_location="cpu"))
    model.eval()
    print(f"✓ Loaded trained policy weights from {model_path}")
else:
    print("Notice: Checkpoint not found. Running random baseline.")

obs, _ = env.reset(seed=123)
step_history = []
total_map_writes = 0
total_food_consumed = 0.0

for step in range(1, 61):
    if not env.agents:
        break
    actions = {}
    with torch.no_grad():
        for a in env.agents:
            obs_t = torch.tensor(obs[a], dtype=torch.float32).unsqueeze(0)
            logits, _ = model(obs_t)
            act = torch.argmax(logits, dim=-1).item()
            actions[a] = act
            if act == WRITE_TO_MAP:
                total_map_writes += 1

    obs, rewards, term, trunc, infos = env.step(actions)
    for a in infos:
        if "food_consumed" in infos[a]:
            total_food_consumed += infos[a]["food_consumed"]

print(f"Rollout Completed (60 steps):")
print(f"- Alive Agents: {len(env.agents)}/{len(env.possible_agents)}")
print(f"- Total Map Writes (Stigmergy Signals): {total_map_writes}")
print(f"- Total Food Units Harvested: {total_food_consumed:.1f}")
print(f"- Non-zero Pheromone Cells on Map: {np.count_nonzero(env.shared_map > 0.05)}")

## 3. Visualizing Resource Pools and Stigmergic Pheromone Trails
Below we plot the final simulation state showing agent positions, resource pool levels, and the spatial pheromone trail matrix.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: Pheromone intensity heatmap
im = axes[0].imshow(env.shared_map, cmap="YlOrRd", vmin=0, vmax=1.0)
axes[0].set_title("Externalized Shared Memory (Pheromone Heatmap)", fontsize=12)
plt.colorbar(im, ax=axes[0], fraction=0.046, pad=0.04)

# Right: World State (Agents & Food Pools)
world_grid = np.zeros((env.grid_size, env.grid_size))
for r in env.rewards_data:
    rx, ry = r["pos"]
    world_grid[ry, rx] = r["amount"] / r["capacity"]
axes[1].imshow(world_grid, cmap="Greens", vmin=0, vmax=1.0)
for a in env.agents:
    ax, ay = env.agent_positions[a]
    axes[1].plot(ax, ay, "ro", markersize=10, label=f"{a}")
axes[1].set_title("World State: Agent Positions & Food Pool Levels", fontsize=12)
axes[1].legend(loc="upper right", bbox_to_anchor=(1.3, 1.0))

plt.tight_layout()
plt.show()

## 4. Empirical Benchmark: Full-Map Visibility vs. Local Stigmergy
The table below shows the empirical results comparing Full-Map global vectors against Local Vision + Stigmergy over 150 training episodes:

| Configuration | Survival Lifespan | Food Consumed | Map Writes / Ep | Cumulative Reward | Survival Rate |
| :--- | :---: | :---: | :---: | :---: | :---: |
| **Full-Map Visibility (Global Vectors)** | 150.0 ± 0.0 steps | 0.7 ± 0.9 units | 129.9 ± 88.3 | 1.4 | **100%** |
| **Local Vision + Stigmergy Memory** | **150.0 ± 0.0 steps** | **46.7 ± 42.8 units** | **21.8 ± 25.2** | **33.9** | **100%** |

### 🎯 Conclusion for Phase 1:
Local Stigmergic Memory successfully solved the collective foraging problem, achieving **>60x higher resource collection efficiency** than the full-map baseline, while demonstrating emergent strategic marking rather than random noise.